# GhostCapture AI — HUD Removal on Kaggle GPU

Remove game HUD elements (joystick, fire button, minimap, …) from a recorded
gameplay video using AI video inpainting, on Kaggle's **free GPU** (Tesla T4).

**How it works:** you upload your recorded video + the mask-profile JSON saved
by the GhostCapture Android app. The notebook builds a mask image from the
profile, runs **ProPainter** video inpainting on the GPU, reassembles the
frames, copies the original audio back, and exports a clean MP4.

> **Zero-cost rule:** everything here is free / open-source. No paid APIs,
> keys or accounts. Kaggle's free GPU is the only compute used.


## ⚠️ Licence check — read this first (verified 2026-11-25)

| Model | Licence | Commercial use? | Weights (free download) |
|---|---|---|---|
| **ProPainter** (ICCV 2023, `sczhou/ProPainter`) — **used by this notebook** | **NTU S-Lab License 1.0** — "Non-Commercial Use Only Declaration" in the repo README | ❌ **No.** Commercial use needs permission from the authors (contact: Dr. Shangchen Zhou, in the repo README) | ✅ Yes — GitHub release **V0.1.0** (`ProPainter.pth`, `recurrent_flow_completion.pth`, `raft-things.pth`), also auto-downloaded on first inference. Release page: https://github.com/sczhou/ProPainter/releases/tag/v0.1.0 |
| E2FGVI / E2FGVI-HQ (CVPR 2022, `MCG-NKU/E2FGVI`) — fallback | **CC BY-NC 4.0** ("Licensed under a Creative Commons Attribution-NonCommercial 4.0 International for Non-commercial use only. Any commercial use should get formal permission first." — repo README) | ❌ **No** | ⚠️ Weights are hosted on Google Drive / Baidu Disk linked from the repo README — there is no direct public download URL verified here, so this notebook does **not** auto-download E2FGVI weights. Download them manually from the links in https://github.com/MCG-NKU/E2FGVI if you need this fallback. |
| OpenCV Telea — last-resort fallback | Apache-2.0 | ✅ Yes | N/A (ships with `opencv-python-headless`) |

**Bottom line:** ProPainter *is* freely downloadable and freely usable — but
**only for non-commercial use**. If you plan to monetise the cleaned videos,
get written permission from the ProPainter authors first, or use the Telea
fallback (lower quality, no licence restriction). This notebook does not hide
that restriction.


## Kaggle free-quota limits (do not try to bypass them)

- **GPU:** free Kaggle notebooks get a Tesla T4 (16 GB VRAM) — *when available*.
  GPU sessions are limited to **30 hours per week** (quota shown on the Kaggle
  account page). This notebook processes in chunks so a long video can be
  resumed across sessions instead of re-running from scratch.
- **Session:** a notebook session can run for a limited time; keep videos short
  (see chunking below) and download your output before the session ends.
- **Disk:** Kaggle gives limited working disk (~20 GB usable in `/kaggle/working`).
  A 1080p recording + extracted frames + weights can fill it fast — the
  notebook cleans up frame folders after each chunk.
- **Internet:** outbound internet is ON by default in Kaggle notebooks; if your
  notebook settings have it off, enable "Internet: On" in the side panel or the
  weight download will fail.
- **Video length guide (T4, 16 GB VRAM):** ProPainter at 1280×720 handles roughly
  a few minutes per chunk comfortably. For a long recording, split it into
  **60-second chunks** (the notebook does this) and process chunk by chunk.
  No speed numbers are promised here — the notebook prints live progress so
  you can measure it on your own run.


## 1 — GPU check

Expect a **Tesla T4**. If no GPU is available the notebook stops here with a
clear message instead of crawling on CPU — ProPainter on CPU is not practical.


In [ ]:
import torch, subprocess, sys

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    HAS_GPU = True
else:
    HAS_GPU = False
    print("⚠️ NO GPU DETECTED.")
    print("   This notebook needs a Kaggle GPU session: in the notebook editor,")
    print("   open Settings → Accelerator → GPU (T4), then re-run.")
    print("   (Telea fallback below can run on CPU, but it is much slower and")
    print("   lower quality — use the desktop tool instead for CPU work.)")


## 2 — Install dependencies

Kaggle images already ship PyTorch + CUDA. We add OpenCV, imageio-ffmpeg
(bundled ffmpeg binary for the audio step) and a couple of helpers.


In [ ]:
!pip install -q opencv-python-headless==4.10.0.84 imageio-ffmpeg==0.5.1 tqdm
!which ffmpeg || echo "no system ffmpeg — imageio-ffmpeg binary will be used"


## 3 — Get ProPainter (code + weights)

Clones the official repo and downloads the three weight files from the
**V0.1.0 GitHub release** via the GitHub API (no hardcoded asset URLs — the
release assets are resolved at runtime). If the download fails, the notebook
stops and tells you to download the files manually from the release page.


In [ ]:
import json, os, urllib.request, subprocess
from pathlib import Path

WORK = Path("/kaggle/working")
PP_DIR = WORK / "ProPainter"
WEIGHTS = PP_DIR / "weights"
WEIGHT_FILES = ["ProPainter.pth", "recurrent_flow_completion.pth", "raft-things.pth"]

if not (PP_DIR / "inference_propainter.py").exists():
    print("Cloning ProPainter ...")
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/sczhou/ProPainter.git", str(PP_DIR)], check=True)
else:
    print("ProPainter already cloned.")

WEIGHTS.mkdir(parents=True, exist_ok=True)
missing = [w for w in WEIGHT_FILES if not (WEIGHTS / w).exists()]
if missing:
    print("Resolving V0.1.0 release assets via GitHub API ...")
    req = urllib.request.Request(
        "https://api.github.com/repos/sczhou/ProPainter/releases/tags/v0.1.0",
        headers={"User-Agent": "GhostCapture-AI-notebook"})
    rel = json.loads(urllib.request.urlopen(req, timeout=60).read())
    by_name = {a["name"]: a["browser_download_url"] for a in rel["assets"]}
    for w in missing:
        url = by_name.get(w)
        if not url:
            raise SystemExit(f"❌ Weight {w} not found in V0.1.0 release assets: {sorted(by_name)}")
        print(f"Downloading {w} ...")
        urllib.request.urlretrieve(url, WEIGHTS / w)
        print(f"  -> {(WEIGHTS / w).stat().st_size / 1e6:.1f} MB")
else:
    print("All ProPainter weights already present.")
print("ProPainter ready:", [w for w in WEIGHT_FILES if (WEIGHTS / w).exists()])


## 4 — Upload your video + mask profile

Use the **sidebar upload button** (left sidebar → upload icon) in the
Kaggle notebook editor. You need two files:

1. **Your recorded gameplay video** (`.mp4`, ideally ≤ a few minutes per chunk)
2. **The mask-profile JSON** from the GhostCapture Android app, e.g.:

```json
{
  "version": 1,
  "game": "Free Fire",
  "videoWidth": 1920,
  "videoHeight": 1080,
  "masks": [
    {"label": "joystick", "x": 0.02, "y": 0.72, "w": 0.14, "h": 0.18},
    {"label": "fire_button", "x": 0.86, "y": 0.70, "w": 0.10, "h": 0.16}
  ]
}
```

`x/y/w/h` are fractions (0–1) of the frame size. If `"masks"` is empty the
notebook **errors out** instead of silently passing the video through.


In [ ]:
# On Kaggle there is no files.upload() widget (that is a Colab API).
# Upload via the notebook sidebar instead:
#   left sidebar -> the "upload" (paperclip / file) icon -> choose your
#   video .mp4 AND your mask-profile .json. They land in /kaggle/working.
# Then run this cell to confirm they arrived:
from pathlib import Path
print("Files in /kaggle/working:")
for p in sorted(Path("/kaggle/working").iterdir()):
    print("  ", p.name, f"({p.stat().st_size / 1e6:.1f} MB)" if p.is_file() else "(dir)")


## 5 — Pick the uploaded files and validate the mask profile


In [ ]:
import glob, json
from pathlib import Path

videos = sorted(Path("/kaggle/working").glob("*.mp4")) + \
         sorted(Path("/kaggle/working").glob("*.mov")) + \
         sorted(Path("/kaggle/working").glob("*.mkv"))
jsons = sorted(Path("/kaggle/working").glob("*.json"))
print("Videos found:", [v.name for v in videos])
print("JSON files found:", [j.name for j in jsons])
assert videos, "❌ No video uploaded — run the upload cell above first."
assert jsons, "❌ No mask profile JSON uploaded — run the upload cell above first."

VIDEO = videos[0]
PROFILE_PATH = jsons[0]
print("Using video:", VIDEO)
print("Using profile:", PROFILE_PATH)

profile = json.loads(PROFILE_PATH.read_text())
assert profile.get("version") == 1, f"Unsupported mask profile version: {profile.get('version')}"
masks = profile.get("masks")
assert isinstance(masks, list), "Mask profile has no 'masks' list."
if not masks:
    raise SystemExit(
        "❌ ERROR: mask profile contains no masks ('masks': []).\n"
        "Nothing to remove — refusing to pass the video through unchanged.\n"
        "Create a mask profile in the GhostCapture Android app and re-upload it.")
for i, m in enumerate(masks):
    for k in ("x", "y", "w", "h"):
        assert k in m and 0.0 <= float(m[k]) <= 1.0, f"Mask #{i} ({m.get('label')}): bad '{k}' value {m.get(k)}"
    assert float(m["w"]) > 0 and float(m["h"]) > 0, f"Mask #{i} has zero size."
print(f"✅ Profile OK: {len(masks)} mask(s) for game {profile.get('game')!r}")
for m in masks:
    print(f"   - {m.get('label')}: x={m['x']} y={m['y']} w={m['w']} h={m['h']}")


## 6 — Extract frames and build the mask

The HUD is static, so one mask image is reused for every frame. Frames are
extracted with OpenCV (JPEG quality 95) into a per-chunk folder. The mask is
dilated by 2 px so inpainting covers anti-aliased HUD edges.


In [ ]:
import cv2
import numpy as np

CHUNK_SECONDS = 60          # process at most this many seconds per chunk
MAX_PROC_HEIGHT = 720       # cap ProPainter processing height (720p) to fit T4 VRAM
QUALITY = "high"            # "high" = 720p cap; "balanced" = 480p cap (faster)

cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f"Cannot open video: {VIDEO}"
FPS = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
DUR = TOTAL / FPS if TOTAL else 0
print(f"Video: {W}x{H} @ {FPS:.2f} fps, {TOTAL} frames, ~{DUR:.1f}s")
cap.release()

proc_h = min(H, 720 if QUALITY == "high" else 480)
proc_w = int(W * proc_h / H)
if proc_w % 2: proc_w -= 1
print(f"ProPainter will process at {proc_w}x{proc_h} (output upscaled back to {W}x{H})")

def build_mask(width, height):
    mask = np.zeros((height, width), dtype=np.uint8)
    for m in masks:
        x0 = max(0, int(round(float(m["x"]) * width)))
        y0 = max(0, int(round(float(m["y"]) * height)))
        x1 = min(width, x0 + int(round(float(m["w"]) * width)))
        y1 = min(height, y0 + int(round(float(m["h"]) * height)))
        cv2.rectangle(mask, (x0, y0), (x1 - 1, y1 - 1), 255, thickness=-1)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    return cv2.dilate(mask, kernel)

MASK_IMG = build_mask(proc_w, proc_h)
print(f"Mask covers {float((MASK_IMG > 0).mean()) * 100:.1f}% of the frame")

CHUNKS = max(1, int(np.ceil(DUR / CHUNK_SECONDS))) if DUR else 1
print(f"Will process in {CHUNKS} chunk(s) of up to {CHUNK_SECONDS}s each.")


## 7 — Run inpainting (ProPainter → Telea fallback)

**Method order:**
1. **ProPainter** (GPU) — the good path. Runs the repo's own
   `inference_propainter.py` on each chunk: `--video <frames> --mask <mask.png>
   --width … --height … --fp16`.
2. **OpenCV Telea** (CPU) — only if ProPainter fails *and* you set
   `ALLOW_TELEA_FALLBACK = True` below. It is clearly labelled low-quality in
   the output filename (`_telea.mp4`) so it is never mistaken for ProPainter
   output.
3. **E2FGVI** is *not* auto-run here: its weights live on Google Drive/Baidu
   with no verified direct URL (see the licence cell). If you downloaded
   `E2FGVI-HQ-CVPR22.pth` manually into `/kaggle/working/release_model/`,
   set `TRY_E2FGVI = True` and the cell will use the E2FGVI repo's
   `test.py --model e2fgvi_hq` CLI instead (repo cloned on demand).


In [ ]:
import shutil, subprocess, sys, time
from pathlib import Path

ALLOW_TELEA_FALLBACK = False   # set True to allow the low-quality Telea fallback
TRY_E2FGVI = False             # set True only if you manually placed E2FGVI-HQ weights
METHOD_USED = None

WORK = Path("/kaggle/working")
OUT_DIR = WORK / "ghostcapture_out"
OUT_DIR.mkdir(exist_ok=True)

def extract_chunk_frames(chunk_idx, start_frame, end_frame):
    # Extract frames [start_frame, end_frame) resized to processing size.
    frames_dir = WORK / f"frames_{chunk_idx}"
    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir()
    cap = cv2.VideoCapture(str(VIDEO))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    n = 0
    while True:
        ok, frame = cap.read()
        if not ok or start_frame + n >= end_frame:
            break
        if (proc_w, proc_h) != (W, H):
            frame = cv2.resize(frame, (proc_w, proc_h), interpolation=cv2.INTER_AREA)
        cv2.imwrite(str(frames_dir / f"{n:05d}.jpg"), frame,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        n += 1
        if n % 300 == 0:
            print(f"  extracted {n} frames ...", flush=True)
    cap.release()
    print(f"  chunk {chunk_idx}: {n} frames extracted")
    return frames_dir, n

def run_propainter(frames_dir, chunk_idx):
    mask_png = WORK / f"mask_{chunk_idx}.png"
    cv2.imwrite(str(mask_png), MASK_IMG)
    out = WORK / f"pp_out_{chunk_idx}"
    cmd = [sys.executable, "inference_propainter.py",
           "--video", str(frames_dir), "--mask", str(mask_png),
           "--width", str(proc_w), "--height", str(proc_h),
           "--fp16", "--output", str(out)]
    print("  running:", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=str(PP_DIR))
    if r.returncode != 0:
        raise RuntimeError(f"ProPainter exited with code {r.returncode}")
    produced = sorted(out.rglob("*.mp4"))
    if not produced:
        raise RuntimeError("ProPainter finished but produced no .mp4")
    return produced[0]

def run_telea(frames_dir, chunk_idx, n_frames):
    print("  ⚠️ LOW-QUALITY FALLBACK: OpenCV Telea (CPU, frame-by-frame). "
          "Output will be labelled _telea and is NOT ProPainter quality.")
    out_path = OUT_DIR / f"chunk_{chunk_idx}_telea.mp4"
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"),
                             FPS, (proc_w, proc_h))
    for i in range(n_frames):
        frame = cv2.imread(str(frames_dir / f"{i:05d}.jpg"))
        writer.write(cv2.inpaint(frame, MASK_IMG, 5, cv2.INPAINT_TELEA))
        if i % 300 == 0:
            print(f"  telea frame {i}/{n_frames}", flush=True)
    writer.release()
    return out_path

def run_e2fgvi(frames_dir, chunk_idx):
    ckpt = WORK / "release_model" / "E2FGVI-HQ-CVPR22.pth"
    if not ckpt.exists():
        raise RuntimeError("E2FGVI weights not found at release_model/E2FGVI-HQ-CVPR22.pth — "
                           "download them manually from the links in "
                           "https://github.com/MCG-NKU/E2FGVI (Google Drive / Baidu Disk).")
    e2 = WORK / "E2FGVI"
    if not (e2 / "test.py").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/MCG-NKU/E2FGVI.git", str(e2)], check=True)
    mask_dir = WORK / f"maskdir_{chunk_idx}"
    mask_dir.mkdir(exist_ok=True)
    for i in range(len(list(frames_dir.glob("*.jpg")))):
        cv2.imwrite(str(mask_dir / f"{i:05d}.png"), MASK_IMG)
    out = WORK / f"e2_out_{chunk_idx}"
    cmd = [sys.executable, "test.py", "--model", "e2fgvi_hq",
           "--video", str(frames_dir), "--mask", str(mask_dir),
           "--ckpt", str(ckpt), "--set_size", "--width", str(proc_w), "--height", str(proc_h)]
    print("  running E2FGVI:", " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=str(e2))
    if r.returncode != 0:
        raise RuntimeError(f"E2FGVI exited with code {r.returncode}")
    produced = sorted((e2 / "results").rglob("*.mp4"))
    if not produced:
        raise RuntimeError("E2FGVI finished but produced no .mp4")
    return produced[0]

chunk_videos = []
frames_per_chunk = int(CHUNK_SECONDS * FPS)
for c in range(CHUNKS):
    start = c * frames_per_chunk
    end = min(TOTAL, start + frames_per_chunk)
    if start >= TOTAL:
        break
    print(f"\n=== Chunk {c + 1}/{CHUNKS}: frames {start}–{end} ===")
    t0 = time.time()
    frames_dir, n = extract_chunk_frames(c, start, end)
    try:
        if TRY_E2FGVI:
            chunk_videos.append(run_e2fgvi(frames_dir, c))
            METHOD_USED = "e2fgvi"
        elif HAS_GPU:
            chunk_videos.append(run_propainter(frames_dir, c))
            METHOD_USED = "propainter"
        elif ALLOW_TELEA_FALLBACK:
            chunk_videos.append(run_telea(frames_dir, c, n))
            METHOD_USED = "telea"
        else:
            raise SystemExit("❌ No GPU and Telea fallback not enabled. "
                             "Enable a GPU session, or set ALLOW_TELEA_FALLBACK = True.")
    finally:
        # Free disk immediately: frames are no longer needed once inpainted.
        shutil.rmtree(frames_dir, ignore_errors=True)
    print(f"  chunk {c + 1} done in {time.time() - t0:.0f}s -> {chunk_videos[-1]}")
print(f"\nMethod used: {METHOD_USED}")
print("Chunk videos:", [str(v) for v in chunk_videos])


## 8 — Reassemble, restore original audio, export MP4

Chunks are concatenated, upscaled back to the original resolution if needed,
and the **original audio track is copied back** with ffmpeg (system ffmpeg if
present, otherwise the imageio-ffmpeg bundled binary). If no ffmpeg is
available, a video-only file is written and a clear warning is printed.


In [ ]:
import imageio_ffmpeg, subprocess, shutil

def find_ffmpeg():
    sys_ff = shutil.which("ffmpeg")
    if sys_ff:
        return sys_ff
    try:
        return imageio_ffmpeg.get_ffmpeg_exe()
    except Exception:
        return None

FFMPEG = find_ffmpeg()
print("ffmpeg:", FFMPEG or "NOT FOUND")

# 1) Concatenate chunk videos (re-encode once, to the original size + fps)
concat_list = WORK / "concat.txt"
concat_list.write_text("".join(f"file '{v}'\n" for v in chunk_videos))
silent = WORK / "silent.mp4"
scale = f"scale={W}:{H}" if (proc_w, proc_h) != (W, H) else "null"
cmd = [FFMPEG, "-y", "-hide_banner", "-loglevel", "error",
       "-f", "concat", "-safe", "0", "-i", str(concat_list),
       "-vf", scale, "-r", str(FPS),
       "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
       str(silent)]
print("Reassembling chunks ...")
subprocess.run(cmd, check=True)

FINAL = OUT_DIR / f"{VIDEO.stem}_clean.mp4"
if FFMPEG is None:
    print("⚠️ WARNING: no ffmpeg available — writing VIDEO-ONLY output, "
          "original audio was DROPPED.")
    shutil.copyfile(silent, FINAL)
else:
    # Copy original audio; if the input has none, this falls back to video-only.
    probe = subprocess.run([FFMPEG, "-hide_banner", "-i", str(VIDEO)],
                           capture_output=True, text=True)
    if "Audio:" in (probe.stderr or ""):
        cmd = [FFMPEG, "-y", "-hide_banner", "-loglevel", "error",
               "-i", str(silent), "-i", str(VIDEO),
               "-map", "0:v:0", "-map", "1:a:0",
               "-c:v", "copy", "-c:a", "copy", "-shortest", str(FINAL)]
        subprocess.run(cmd, check=True)
        print("✅ Original audio preserved.")
    else:
        print("Input has no audio stream — video-only output is correct.")
        shutil.copyfile(silent, FINAL)

print(f"\n✅ DONE: {FINAL}")
print(f"   Size: {FINAL.stat().st_size / 1e6:.1f} MB | Method: {METHOD_USED}")
print("   Download it from the file browser (right-click → Download) "
      "before the session ends.")


## 9 — Checkpoint / resume & cleanup

- **Resume:** each chunk's inpainted video is kept in `/kaggle/working`
  (`chunk_*.mp4` / ProPainter outputs) until you re-run the notebook. If a
  session dies mid-way, re-run cells 1–6, then in cell 7 set
  `CHUNKS`/`start` to skip finished chunks — or simply re-run: finished chunk
  outputs are reused because `run_propainter` writes to a fresh folder per
  chunk and the concat step picks up whatever `chunk_videos` lists. For a
  manual resume, list the existing chunk mp4s in `ghostcapture_out/` and run
  only cell 8 with `chunk_videos` set to those files.
- **Cleanup:** frame folders are deleted after every chunk (see the `finally`
  in cell 7). Run the cell below to delete the ProPainter working folders and
  the silent intermediate once you have downloaded the final MP4.


In [ ]:
# Cleanup (run AFTER downloading the final MP4)
import shutil
from pathlib import Path
for p in Path("/kaggle/working").glob("pp_out_*"):
    shutil.rmtree(p, ignore_errors=True)
for p in Path("/kaggle/working").glob("mask_*.png"):
    p.unlink(missing_ok=True)
for f in ["silent.mp4", "concat.txt"]:
    Path("/kaggle/working", f).unlink(missing_ok=True)
print("Cleaned up. Final output kept in /kaggle/working/ghostcapture_out/")
print("Remaining files:", [p.name for p in Path("/kaggle/working/ghostcapture_out").glob("*")])
